In [3]:
import requests
from bs4 import BeautifulSoup
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import dask.dataframe as dd
import time
import os

page = requests.get("https://www.worldometers.info/coronavirus")
#print(page.content)
# Soup is a variable containing the HTML of the webpage
soup = BeautifulSoup(page.content, 'lxml')

# Search for the table and extracting it
table = soup.find('table', attrs={'id': 'main_table_countries_today'})
#print(table)
rows = table.find_all("tr", attrs={"style": ""})
#print(rows)
data = []
for i, item in enumerate(rows):
    if i == 0:
        data.append(item.text.strip().split("\n")[:16])
    else:
        data.append(item.text.strip().split("\n")[:15])

data[0][13] = data[0][13] + data[0][14]
data[0][14] = 'Population'
data[1].pop()
data[1].insert(0, ' ')
#print(data)
#dt = pd.DataFrame(data)
dt = pd.DataFrame(data[1:], columns=data[0][:15]) # Formatting the header
df = dd.from_pandas(dt, npartitions=1)
#print(df.head())
today = time.strftime("%d-%m-%Y")
print(today)
path = "Extracted_data"
if not os.path.exists(path):
    os.makedirs(path)
df.compute().to_csv(path + "\\data_covid19_{}.csv".format(today))

22-04-2022
